# Synax — train LLM tiếng Việt từ số 0

Notebook này chạy trên Colab Pro (L4/A100 nếu có). Toàn bộ checkpoint ghi vào Google Drive.

Đọc trước: `docs/00-plan.md` và `docs/01-data.md`.


In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

REPO = "/content/Synax"
if not os.path.exists(REPO):
    !git clone https://github.com/wiothemilo-lang/Synax.git {REPO}
os.chdir(REPO)
!pip install -q -e .


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/synax"
os.makedirs(DRIVE, exist_ok=True)
print("drive:", DRIVE)


### Tùy chọn: tải mẫu Wikipedia tiếng Việt

Nếu chưa có corpus riêng, cell dưới tải 2.000 bài Wikipedia tiếng Việt (CC BY-SA)
để chạy thử toàn bộ pipeline. Bỏ qua nếu đã upload `MyDrive/synax/raw/corpus.txt`.


In [ ]:
from pathlib import Path

# Tùy chọn: tải mẫu Wikipedia tiếng Việt (CC BY-SA)
!pip install -q datasets
from datasets import load_dataset

RAW = f"{DRIVE}/raw/corpus.txt"
Path(RAW).parent.mkdir(parents=True, exist_ok=True)
if not Path(RAW).exists():
    ds = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train[:2000]")
    with open(RAW, "w", encoding="utf-8") as handle:
        for row in ds:
            handle.write(row["text"] + "\n\n")
    print("wrote", RAW)
else:
    print("corpus already exists:", RAW)


## 1. Dữ liệu

Upload corpus sạch lên `MyDrive/synax/raw/corpus.txt` (hoặc `.jsonl`).

Nếu chưa có, chạy pipeline ở máy local trước:

```bash
python -m synax.data.pipeline --input raw/*.txt --output data/clean/corpus.jsonl
```


In [ ]:
from pathlib import Path

RAW = f"{DRIVE}/raw/corpus.txt"
TOKENIZER = f"{DRIVE}/tokenizer/sp32"
Path(TOKENIZER).parent.mkdir(parents=True, exist_ok=True)

if not Path(TOKENIZER + ".model").exists():
    !python -m synax.data.tokenizer train --input {RAW} --output-prefix {TOKENIZER} --vocab-size 32000 --tmp-dir {DRIVE}
else:
    print("tokenizer already exists:", TOKENIZER + ".model")


In [ ]:
TRAIN_BIN = f"{DRIVE}/tokenized/train.bin"
VAL_BIN = f"{DRIVE}/tokenized/val.bin"
Path(TRAIN_BIN).parent.mkdir(parents=True, exist_ok=True)

if not Path(TRAIN_BIN).exists():
    !python -m synax.data.tokenizer encode --input {RAW} --tokenizer {TOKENIZER}.model --output {TRAIN_BIN} --val-output {VAL_BIN} --val-ratio 0.005
else:
    print("shards already exist")


## 2. Chạy thử nhanh (200 step)

Dùng để kiểm tra toàn bộ luồng trên Colab. Với corpus mẫu vài MB, 200 step là đủ.
Checkpoint ghi vào `checkpoints/60m-test`.


In [ ]:
OUT_TEST = f"{DRIVE}/checkpoints/60m-test"
Path(OUT_TEST).mkdir(parents=True, exist_ok=True)

!python -m synax.train.pretrain \
  --config configs/model-60m.yaml \
  --train-bin {TRAIN_BIN} \
  --val-bin {VAL_BIN} \
  --tokenizer {TOKENIZER}.model \
  --out-dir {OUT_TEST} \
  --steps 200


## 3. Chạy đầy đủ

Chỉ chạy cell này khi đã có **~1B token** (4–6 GB văn bản sạch).
Session có thể ngắt bất cứ lúc nào: chạy lại cell với `--resume auto` là tiếp tục.


In [ ]:
OUT = f"{DRIVE}/checkpoints/60m"
Path(OUT).mkdir(parents=True, exist_ok=True)

!python -m synax.train.pretrain \
  --config configs/model-60m.yaml \
  --train-bin {TRAIN_BIN} \
  --val-bin {VAL_BIN} \
  --tokenizer {TOKENIZER}.model \
  --out-dir {OUT} \
  --resume auto


In [ ]:
from pathlib import Path

RUN_DIR = Path(f"{DRIVE}/checkpoints/60m")
if not (RUN_DIR / "latest.txt").exists():
    RUN_DIR = Path(f"{DRIVE}/checkpoints/60m-test")
latest = (RUN_DIR / "latest.txt").read_text().strip()
print("checkpoint:", RUN_DIR / latest)

!python -m synax.inference.generate \
  --checkpoint {RUN_DIR}/{latest} \
  --tokenizer {TOKENIZER}.model \
  --prompt "Việt Nam là" \
  --max-new-tokens 120


## Ghi chú

- Session Colab có thể ngắt: chạy lại cell train với `--resume auto` là tiếp tục.
- Checkpoint giữ 2 bản mới nhất, bản cũ tự xoá.
- Không đưa `MyDrive/synax` vào git.
- Kaggle: clone repo, attach dataset, chạy cùng lệnh với `--resume auto`.
